# ДЗ 5. Первый взгляд на orders.csv в Pandas
Срок: **до воскресенья 22.11, 24:00**.
Сдача: папка `hw05` репозитория `dsa-homework` (`hw05.ipynb`): `git add hw05` → `git commit` → `git push`, ссылку прикрепите в LMS.

Каждый ответ: **число и вывод для менеджера** (в print или Markdown-ячейке). Менеджер не читает код: вывод отвечает, что видно в данных и что делать.

Инструменты: Pandas с урока 5 и всё, что было раньше. `groupby`, `apply`, `.str`-методы (кроме готовой строки для `amount`) пока не используем. Если берёте непройденный инструмент, объясните его в комментарии.

- **[База]** обязательно для всех · **[Стандарт]** сколько получится · **[Челлендж]** для тех, кому легко.

## [Для всех] Чтение кода: без запуска
Сначала напишите ответ и «почему», потом запустите.
```python
import pandas as pd
# 1
s = pd.Series([10, 20, 30], index=["a", "b", "c"])
print(s["b"], s.iloc[0])
# 2
a = pd.Series({"Алматы": 5, "Шымкент": 2})
b = pd.Series({"Шымкент": 1, "Астана": 4})
print(a + b)
# 3
df = pd.DataFrame({"x": [1, 2, 3], "y": [4, 5, 6]})
print(df[df["x"] > 1].shape)
# 4
print(type(df["x"]).__name__, type(df[["x"]]).__name__)
# 5
f = df[df["x"] > 1]
print(f.index.tolist(), f.iloc[0, 1])
```

Ваши ответы и объяснения:

1.s['b'] = 20 (ищет по ярлыку 'b'), s.iloc[0] = 10 (ищет по позиции 0)

2.Алматы     NaN
  Астана     NaN
  Шымкент    3.0
  pandas складывает по ярлыкам. Шымкент есть в обеих Series, поэтому 2 + 1 = 3. Алматы и Астана есть только в одной из них, и получается NaN. Из-за NaN тип становится float. Индекс в результате отсортирован.

3. (2, 2)
Почему: условию x > 1 подходят две строки (x = 2 и 3), а столбцов по-прежнему два.

4. Series DataFrame
Почему: одна пара скобок с одним именем возвращает Series. Двойные скобки — это список столбцов, и даже из одного столбца получается DataFrame.

5. [1, 2] 5
Почему: после фильтрации строки сохраняют старый индекс 1 и 2, а не перенумеровываются с 0. iloc[0, 1] берёт первую по позиции строку (x = 2, y = 5) и второй по позиции столбец y, то есть 5.

In [34]:
# Загрузка orders.csv: в Colab нажмите «Выбрать файлы» и укажите orders.csv из LMS.
# Если файл уже лежит рядом с ноутбуком, загрузка пропускается.
import os
if not os.path.exists("orders.csv"):
    try:
        from google.colab import files
        files.upload()
    except ImportError:
        print("Положите orders.csv в одну папку с ноутбуком")
print("orders.csv на месте:", os.path.exists("orders.csv"))

orders.csv на месте: True


In [35]:
import pandas as pd
import numpy as np

df = pd.read_csv("orders.csv")
# Готовая строка: суммы вида "15 990" записаны текстом. Убираем пробел и переводим в числа (подробно на уроке 6)
df["amount"] = pd.to_numeric(df["amount"].str.replace(" ", ""))
print(df.shape)
print(df.head())

(5463, 8)
   order_id        order_date customer_id       city category   amount  \
0    101674  2026-09-01 08:00      C00681     Алматы   Одежда   8070.0   
1    101214  2026-09-01 08:00      C01210     Астана  Красота   3630.0   
2    102424  2026-09-01 08:00      C01408  Караганда      Дом   9120.0   
3    102333  2026-09-01 08:00      C01529     Алматы   Одежда   9350.0   
4    102564  2026-09-01 09:00      C00657     Астана      Дом  11010.0   

   delivery_days   status  
0            1.0  оплачен  
1            2.0  оплачен  
2            3.0  оплачен  
3            2.0  оплачен  
4            1.0  оплачен  


In [36]:
# Даты записаны текстом "2026-09-01 08:00", поэтому строки можно сравнивать
sep = df["order_date"] < "2026-10"     # сентябрь
octo = ~sep                            # октябрь
paid = df["status"] == "оплачен"
not_test = df["customer_id"] != "TEST"
SH = ["Шымкент", "шымкент", "Shymkent"]
print("сентябрь:", sep.sum(), "| октябрь:", octo.sum())

сентябрь: 2782 | октябрь: 2681


## [База] 1. Паспорт таблицы
Сколько строк и столбцов? Какие типы у столбцов? Сколько пропусков в `delivery_days`? Сколько разных значений в `city`, `category`, `status`, `customer_id`?
Напишите менеджеру три проблемы данных, которые видно уже сейчас, и чем каждая опасна для отчёта о выручке.

In [37]:
# 1
rows, columns = df.shape[0], df.shape[1]
print(f'Кол - во строк: {rows}, кол - во столбцов: {columns}')
print(f'Типы столбцов: {df.dtypes}')
print(f'Кол - во пропусков в delivery days: {df['delivery_days'].isna().sum()}')
print(f'Кол - во различных значений:')
print(f'В city: {df['city'].nunique()}, в category : {df['category'].nunique()}, в status: {df['status'].nunique()}, в customer_id: {df['customer_id'].nunique()}')
#Города записаны по-разному («Алматы», «алматы», «Almaty», « Алматы» и т.д.).
#Повторяющиеся order_id. Один и тот же заказ записан несколько раз, и при суммировании его выручка учитывается дважды. Итоговая выручка получается завышенной.
#Тестовые заказы (TEST) в данных. Это не реальные продажи. Если их не исключить, выручка будет завышена.


Кол - во строк: 5463, кол - во столбцов: 8
Типы столбцов: order_id           int64
order_date        object
customer_id       object
city              object
category          object
amount           float64
delivery_days    float64
status            object
dtype: object
Кол - во пропусков в delivery days: 160
Кол - во различных значений:
В city: 14, в category : 5, в status: 2, в customer_id: 2003



## [База] 2. Типичный заказ
Для оплаченных заказов без TEST: среднее, медиана, 25-й и 75-й перцентили суммы.
Какой чек «типичный»? Что честнее показать в отчёте, среднее или медиану, и почему?

In [38]:
# 2
mean1 = df.loc[paid & not_test, 'amount'].mean()
median1 = df.loc[paid & not_test, 'amount'].median()
per25 = df.loc[paid & not_test, 'amount'].quantile(0.25)
per75 = df.loc[paid & not_test, 'amount'].quantile(0.75)
print(f'Среднее : {mean1}, медиана : {median1}, 25-й перцентиль : {per25}, 75-й {per75}')

#Типичный чек - около 10 300. Половина заказов лежит в диапазоне примерно от 5 900 до 18 700.
#Что честнее показать в отчёте — медиану. Среднее примерно на 40% выше медианы
#Медиана к выбросам устойчива и лучше отражает обычный заказ. Хорошо дополнить её диапазоном между 25-м и 75-м перцентилями.

Среднее : 14672.391348704346, медиана : 10300.0, 25-й перцентиль : 5880.0, 75-й 18690.0


## [База] 3. Отмены по городам
Доля отменённых заказов: по всем заказам, в Шымкенте, в Алматы (все написания каждого города через `isin`).
Какой город выделяется и во сколько раз?

In [39]:
#3
print(df['city'].value_counts())

shym = df['city'].isin(['Шымкент', 'шымкент', 'Shymkent'])
ala  = df['city'].isin(['Алматы', ' Алматы', 'алматы', 'АЛМАТЫ', 'Almaty'])

all_orders = (~paid).mean()
shym_orders = (~paid)[shym].mean()
ala_orders = (~paid)[ala].mean()
print()
print(f'Доля отмененных заказов в целом: {all_orders}, в Шымкенте: {shym_orders}, в Алмате: {ala_orders}')

#Выделяется Шымкент. Доля отмен там примерно в 2 раза выше средней (20.4 / 10.2 ≈ 2.0) и примерно в 2.6 раза выше, чем в Алматы (20.4 / 7.9 ≈ 2.6).

city
Алматы       1927
Астана       1226
Шымкент       711
Караганда     538
Актобе        397
Атырау        146
астана         84
Astana         80
Almaty         69
 Алматы        66
алматы         64
АЛМАТЫ         62
шымкент        54
Shymkent       39
Name: count, dtype: int64

Доля отмененных заказов в целом: 0.1023247300018305, в Шымкенте: 0.20398009950248755, в Алмате: 0.07861060329067641


## [Стандарт] 4. Октябрь против сентября по двум городам
Выручка оплаченных заказов без TEST за сентябрь и октябрь в Шымкенте и в Алматы, изменение в %.
Дубли пока не убираем. Где падение сильнее и во сколько раз? Что сказать менеджеру?

In [40]:
# 4
shym = df['city'].isin(['Шымкент', 'шымкент', 'Shymkent'])
ala = df['city'].isin(['Алматы', ' Алматы', 'алматы', 'АЛМАТЫ', 'Almaty'])

revenue_shym_sep = df.loc[paid & shym & sep & not_test, 'amount'].sum()
revenue_shym_oct = df.loc[paid & shym & octo & not_test, 'amount'].sum()

revenue_ala_sep = df.loc[paid & ala & sep & not_test, 'amount'].sum()
revenue_ala_oct = df.loc[paid & ala & octo & not_test, 'amount'].sum()

pct_shym = (revenue_shym_oct / revenue_shym_sep - 1) * 100
pct_ala  = (revenue_ala_oct  / revenue_ala_sep  - 1) * 100

print(f'Шымкент: сентябрь {revenue_shym_sep:,.0f}, октябрь {revenue_shym_oct:,.0f}, изменение {pct_shym:+.1f}%')
print(f'Алматы:  сентябрь {revenue_ala_sep:,.0f}, октябрь {revenue_ala_oct:,.0f}, изменение {pct_ala:+.1f}%')
print(f'Падение в Шымкенте сильнее в {pct_shym / pct_ala:.1f} раза')

#В Шымкенте выручка в октябре просела заметно сильнее, чем в Алматы: −18% против −6%. Это стоит разобрать отдельно, например посмотреть, упало ли число заказов или средний чек, и в каких категориях.
#Но цифры предварительные: дубли ещё не убраны, поэтому выручка может быть немного завышена

Шымкент: сентябрь 4,949,800, октябрь 4,044,010, изменение -18.3%
Алматы:  сентябрь 15,523,940, октябрь 14,643,020, изменение -5.7%
Падение в Шымкенте сильнее в 3.2 раза


## [Стандарт] 5. Исправьте код от AI
AI написал код «отменённые заказы Шымкента». В нём **две** ошибки: одна роняет код, вторая тихо даёт неверное число. Найдите обе и исправьте.
```python
cancel_sh = df[df["city"] == "Шымкент" & df["status"] == "отменён"]
print(len(cancel_sh))
```

In [41]:
# 5
cancel_sh = (~paid)[shym].sum()
print(cancel_sh)
#Роняет код: у & приоритет выше, чем у ==, поэтому Python пытается вычислить "Шымкент" & df["status"] и падает. Каждое условие надо взять в скобки.
#Тихо даёт неверное число: == "Шымкент" ловит только одно написание, а шымкент и Shymkent теряются. Нужен isin.


164


## [Стандарт] 6. Самые частые клиенты
Кто сделал больше всего заказов (`value_counts` по `customer_id`)? Покажите все заказы этого клиента (`loc` с маской, столбцы дата, город, сумма, статус).
Есть ли среди его заказов повторяющиеся `order_id`? Что это значит для выручки, которую мы считали весь урок?

In [42]:
# 6
most_cust = df['customer_id'].value_counts().head(1).index[0]


all_orders = df.loc[df['customer_id'] == most_cust, ['order_id', 'order_date', 'city', 'amount', 'status']]
duplicate = all_orders.duplicated(keep=False)


print(f'Больше всего заказов сделал: {most_cust}')
print('Все его заказы:')
print(all_orders)
print(f'Повторяющиеся заказы: {duplicate.sum()}')

# Дубликат посчитан дважды, поэтому только у этого клиента выручка завышена на 8890. Если дубли есть и у других клиентов, вся выручка, которую мы считали весь урок, завышена

Больше всего заказов сделал: C01818
Все его заказы:
      order_id        order_date       city   amount   status
57      101125  2026-09-01 15:00    Шымкент   7350.0  оплачен
779     100096  2026-09-09 18:00    Шымкент  45270.0  оплачен
1371    101504  2026-09-15 21:00     Алматы   8890.0  оплачен
1375    101504  2026-09-15 21:00     Алматы   8890.0  оплачен
2195    101860  2026-09-24 18:00     Астана   1490.0  оплачен
2617    101462  2026-09-29 09:00     Астана  22170.0  оплачен
2776    101495  2026-09-30 21:00     Алматы  20180.0  оплачен
3417    103416  2026-10-08 14:00  Караганда   4330.0  оплачен
4279    102644  2026-10-18 08:00     Астана  13600.0  оплачен
4610    104001  2026-10-21 19:00     Алматы  21990.0  оплачен
4778    104572  2026-10-23 18:00     Алматы  11140.0  оплачен
Повторяющиеся заказы: 2


## [Челлендж] 7. Выбросы по правилу IQR
Для оплаченных заказов без TEST: `IQR = Q3 − Q1`, граница = `Q3 + 1.5 * IQR`. Сколько заказов выше границы, какая это доля и какая у них категория чаще всего?
Сравните с ДЗ 4: по z-оценке выбросов было 87. Почему правила дают разное число?

In [43]:
# 7
base = paid & not_test
q3 = df.loc[paid & not_test, 'amount'].quantile(0.75)
q1 = df.loc[paid & not_test, 'amount'].quantile(0.25)
boundary = q3 + 1.5 * (q3 - q1)
above_boundary = base & (df['amount'] > boundary)
all_ord = (paid) & not_test
cat_above_boundary = df.loc[above_boundary, 'category']


print(f'Заказы выше границы: {above_boundary.sum()}, их доля: {above_boundary.sum()/all_ord.sum() * 100}, частая категория : {cat_above_boundary.value_counts().head(1).index[0]}')

# Z-оценка считается через среднее и стандартное отклонение, а их сильно раздувают сами выбросы.
# Из-за этого граница уезжает вверх, и выбросов находится меньше (87). IQR опирается на квартили, которые к выбросам устойчивы, поэтому граница ниже и выбросов больше (320).
# К тому же суммы скошены вправо, а z-оценка рассчитана на нормальное распределение.


Заказы выше границы: 320, их доля: 6.529279738828811, частая категория : Электроника


## [Челлендж] 8. Доля отмен для каждого написания города одной строкой
Поделите две Series: число отменённых заказов по `city` и число всех заказов по `city` (обе через `value_counts`). Отсортируйте.
Pandas делит по ярлыкам: проверьте это. Доля отмен у «Шымкент», «шымкент» и «Shymkent» сильно разная. Сколько заказов у каждого написания?
Можно ли судить о городе по одному написанию? Что нужно сделать с городами до анализа (это тема урока 6)?

In [44]:
# 8
by_city_cancelled = df.loc[~paid, 'city'].value_counts()
by_city_all = df['city'].value_counts()


cancel_rate = (by_city_cancelled / by_city_all).fillna(0).sort_values(ascending=False)
print(cancel_rate)


m = df['city'] == 'Шымкент'
print(cancel_rate['Шымкент'], (~paid & m).sum() / m.sum())

spellings = ['Шымкент', 'шымкент', 'Shymkent']

print(by_city_all[spellings])
print(cancel_rate[spellings])

city
Shymkent     0.230769
Шымкент      0.215190
алматы       0.140625
астана       0.130952
Актобе       0.103275
Astana       0.100000
Караганда    0.089219
Астана       0.084829
Алматы       0.079917
Атырау       0.075342
Almaty       0.057971
 Алматы      0.045455
шымкент      0.037037
АЛМАТЫ       0.032258
Name: count, dtype: float64
0.21518987341772153 0.21518987341772153
city
Шымкент     711
шымкент      54
Shymkent     39
Name: count, dtype: int64
city
Шымкент     0.215190
шымкент     0.037037
Shymkent    0.230769
Name: count, dtype: float64
